In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------
# Basic Residual Block (1D)
# -----------------------------
class BasicBlock1d(nn.Module):
    expansion = 1

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=7, stride=stride, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=7, stride=1, padding=3, bias=False)
        self.bn2 = nn.BatchNorm1d(out_ch)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_ch)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out += identity
        out = F.relu(out)
        return out


# -----------------------------
# Main Model
# -----------------------------
class ResNet1dWithTabularOriginal(nn.Module):
    def __init__(self, num_tabular_features=7, num_outputs=12):
        super().__init__()

        # ---- Stem ----
        self.conv1 = nn.Conv1d(12, 16, kernel_size=15, stride=2, padding=7, bias=False)
        self.bn1 = nn.BatchNorm1d(16)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

        # ---- Residual Stages ----
        self.layer1 = self._make_layer(16, 16, blocks=3, stride=1)
        self.layer2 = self._make_layer(16, 32, blocks=4, stride=2)
        self.layer3 = self._make_layer(32, 64, blocks=6, stride=2)
        self.layer4 = self._make_layer(64, 128, blocks=3, stride=2)

        # ---- Global Pooling ----
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.maxpool_global = nn.AdaptiveMaxPool1d(1)

        # ---- Regularization ----
        self.dropout = nn.Dropout(0.5)

        # ---- Final Fusion Layer ----
        self.fc = nn.Linear(128 * 2 + num_tabular_features, num_outputs)

    def _make_layer(self, in_ch, out_ch, blocks, stride):
        layers = []
        layers.append(BasicBlock1d(in_ch, out_ch, stride))
        for _ in range(1, blocks):
            layers.append(BasicBlock1d(out_ch, out_ch))
        return nn.Sequential(*layers)

    def forward(self, ecg, tabular):
        """
        ecg: [B, 12, 2500]
        tabular: [B, 7]
        """
        x = self.relu(self.bn1(self.conv1(ecg)))
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        avg = self.avgpool(x)
        mx = self.maxpool_global(x)
        x = torch.cat([avg, mx], dim=1)   # [B, 256, 1]
        x = x.squeeze(-1)                 # [B, 256]

        x = self.dropout(x)

        x = torch.cat([x, tabular], dim=1)  # [B, 263]
        out = self.fc(x)

        return out


In [5]:
model_with_tabular = ResNet1dWithTabularOriginal()
#Print a summary
from torchinfo import summary

#summary(model, input_size=(1,2500,12), col_names=["input_size", "output_size", "num_params", "params_percent", "kernel_size", "mult_adds", "trainable"])
summary(model_with_tabular)

Layer (type:depth-idx)                   Param #
ResNet1dWithTabularOriginal              --
├─Conv1d: 1-1                            2,880
├─BatchNorm1d: 1-2                       32
├─ReLU: 1-3                              --
├─MaxPool1d: 1-4                         --
├─Sequential: 1-5                        --
│    └─BasicBlock1d: 2-1                 --
│    │    └─Conv1d: 3-1                  1,792
│    │    └─BatchNorm1d: 3-2             32
│    │    └─Conv1d: 3-3                  1,792
│    │    └─BatchNorm1d: 3-4             32
│    │    └─Sequential: 3-5              --
│    └─BasicBlock1d: 2-2                 --
│    │    └─Conv1d: 3-6                  1,792
│    │    └─BatchNorm1d: 3-7             32
│    │    └─Conv1d: 3-8                  1,792
│    │    └─BatchNorm1d: 3-9             32
│    │    └─Sequential: 3-10             --
│    └─BasicBlock1d: 2-3                 --
│    │    └─Conv1d: 3-11                 1,792
│    │    └─BatchNorm1d: 3-12            32
│    │   

### Now I'll create another model that receives tabular data optinally only, so that I can test it with other datasets

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# -----------------------------
# Basic Residual Block (1D)
# -----------------------------
class BasicBlock1d(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=7, stride=stride, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=7, stride=1, padding=3, bias=False)
        self.bn2 = nn.BatchNorm1d(out_ch)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_ch)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        return F.relu(out)


# -----------------------------
# Main Model
# -----------------------------
class ResNet1dWithTabular(nn.Module):
    def __init__(
        self,
        num_outputs=12,
        num_tabular_features=7,
        use_tabular=True,
        dropout_p=0.5
    ):
        super().__init__()
        self.use_tabular = use_tabular
        self.num_tabular_features = num_tabular_features

        # ---- Stem ----
        self.conv1 = nn.Conv1d(12, 16, kernel_size=15, stride=2, padding=7, bias=False)
        self.bn1 = nn.BatchNorm1d(16)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

        # ---- Residual Stages ----
        self.layer1 = self._make_layer(16, 16, blocks=3, stride=1)
        self.layer2 = self._make_layer(16, 32, blocks=4, stride=2)
        self.layer3 = self._make_layer(32, 64, blocks=6, stride=2)
        self.layer4 = self._make_layer(64, 128, blocks=3, stride=2)

        # ---- Global Pooling ----
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.maxpool_global = nn.AdaptiveMaxPool1d(1)

        # ---- Dropout ----
        self.dropout = nn.Dropout(dropout_p)

        # ---- Final Layer Size ----
        ecg_feature_dim = 128 * 2  # avg + max pool

        if self.use_tabular:
            fc_input_dim = ecg_feature_dim + num_tabular_features
        else:
            fc_input_dim = ecg_feature_dim

        self.fc = nn.Linear(fc_input_dim, num_outputs)

    def _make_layer(self, in_ch, out_ch, blocks, stride):
        layers = [BasicBlock1d(in_ch, out_ch, stride)]
        for _ in range(1, blocks):
            layers.append(BasicBlock1d(out_ch, out_ch))
        return nn.Sequential(*layers)

    def forward(self, ecg, tabular=None):
        """
        ecg: [B, 12, 2500]
        tabular: [B, num_tabular_features] OR None
        """

        # ---- ECG Backbone ----
        x = self.relu(self.bn1(self.conv1(ecg)))
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        avg = self.avgpool(x)
        mx = self.maxpool_global(x)
        x = torch.cat([avg, mx], dim=1)   # [B, 256, 1]
        x = x.squeeze(-1)                 # [B, 256]
        x = self.dropout(x)

        # ---- Fusion (optional) ----
        if self.use_tabular:
            if tabular is None:
                raise ValueError("Model was created with use_tabular=True but no tabular data was provided.")
            x = torch.cat([x, tabular], dim=1)

        out = self.fc(x)
        return out


In [9]:
#Instantiate the model
model = ResNet1dWithTabular(use_tabular=False, num_outputs=12)
summary(model)

Layer (type:depth-idx)                   Param #
ResNet1dWithTabular                      --
├─Conv1d: 1-1                            2,880
├─BatchNorm1d: 1-2                       32
├─ReLU: 1-3                              --
├─MaxPool1d: 1-4                         --
├─Sequential: 1-5                        --
│    └─BasicBlock1d: 2-1                 --
│    │    └─Conv1d: 3-1                  1,792
│    │    └─BatchNorm1d: 3-2             32
│    │    └─Conv1d: 3-3                  1,792
│    │    └─BatchNorm1d: 3-4             32
│    │    └─Sequential: 3-5              --
│    └─BasicBlock1d: 2-2                 --
│    │    └─Conv1d: 3-6                  1,792
│    │    └─BatchNorm1d: 3-7             32
│    │    └─Conv1d: 3-8                  1,792
│    │    └─BatchNorm1d: 3-9             32
│    │    └─Sequential: 3-10             --
│    └─BasicBlock1d: 2-3                 --
│    │    └─Conv1d: 3-11                 1,792
│    │    └─BatchNorm1d: 3-12            32
│    │   

In [10]:
# Load saved arrays from checkpoints/XY_arrays_10s.npz into kernel variables
import os
import numpy as np

npz_path = os.path.join('checkpoints', 'XY_arrays_10s.npz')
if not os.path.exists(npz_path):
    raise FileNotFoundError(f"Dataset file not found: {npz_path}")

with np.load(npz_path, allow_pickle=False) as data:
    X_train = data['X_train']
    X_valid = data['X_valid']
    X_test  = data['X_test']
    Y_train = data['Y_train']
    Y_valid = data['Y_valid']
    Y_test  = data['Y_test']

print('Loaded arrays from:', npz_path)
print('Shapes:')
print('  X_train:', np.shape(X_train))
print('  X_valid:', np.shape(X_valid))
print('  X_test :', np.shape(X_test))
print('  Y_train:', np.shape(Y_train))
print('  Y_valid:', np.shape(Y_valid))
print('  Y_test :', np.shape(Y_test))


Loaded arrays from: checkpoints\XY_arrays_10s.npz
Shapes:
  X_train: (14370, 1000, 12)
  X_valid: (1772, 1000, 12)
  X_test : (1812, 1000, 12)
  Y_train: (14370, 9)
  Y_valid: (1772, 9)
  Y_test : (1812, 9)


In [11]:
# Prepare data tensors and loaders; handle shape [N, 12, 2500] vs [N, 2500, 12]
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# Ensure ECG arrays have shape [N, 12, 2500]

def ensure_ecg_ch_last_to_ch_first(arr: np.ndarray) -> np.ndarray:
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D ECG array, got shape {arr.shape}")
    if arr.shape[1] == 12 and arr.shape[2] == 2500:
        return arr
    if arr.shape[1] == 2500 and arr.shape[2] == 12:
        return np.transpose(arr, (0, 2, 1))
    raise ValueError(f"Unexpected ECG shape {arr.shape}; expected [N, 12, 2500] or [N, 2500, 12]")

X_train_ecg = ensure_ecg_ch_last_to_ch_first(X_train)
X_valid_ecg = ensure_ecg_ch_last_to_ch_first(X_valid)

# Dataset / DataLoader
class ECGDataset(Dataset):
    def __init__(self, x: np.ndarray, y: np.ndarray):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return self.x.shape[0]
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

train_ds = ECGDataset(X_train_ecg, Y_train)
valid_ds = ECGDataset(X_valid_ecg, Y_valid)

batch_size = 64
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False)

# Class weights for BCEWithLogitsLoss (pos_weight)
pos_counts = Y_train.sum(axis=0)
neg_counts = Y_train.shape[0] - pos_counts
pos_weight = (neg_counts + 1e-6) / (pos_counts + 1e-6)
pos_weight_t = torch.tensor(pos_weight, dtype=torch.float32)

n_classes = Y_train.shape[1]
print('Prepared loaders:')
print('  X_train_ecg:', X_train_ecg.shape, 'Y_train:', Y_train.shape)
print('  X_valid_ecg:', X_valid_ecg.shape, 'Y_valid:', Y_valid.shape)
print('  n_classes:', n_classes)


ValueError: Unexpected ECG shape (14370, 1000, 12); expected [N, 12, 2500] or [N, 2500, 12]

In [ ]:
# Train ResNet1dWithTabular (multilabel) with BCEWithLogitsLoss and pos_weight
import torch
from torch import nn
from torch.optim import AdamW

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_outputs = int(n_classes)

model = ResNet1dWithTabular(use_tabular=False, num_outputs=num_outputs).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t.to(device))
optimizer = AdamW(model.parameters(), lr=1e-3)

epochs = 10
best_val = float('inf')
ckpt_dir = 'checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)
best_path = os.path.join(ckpt_dir, 'resnet_10s_best.pth')

for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(ecg=xb, tabular=None)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
    avg_train = total_loss / len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in valid_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(ecg=xb, tabular=None)
            loss = criterion(logits, yb)
            val_loss += loss.item() * xb.size(0)
    avg_val = val_loss / len(valid_loader.dataset)

    print(f"Epoch {epoch:02d} | train_loss: {avg_train:.4f} | val_loss: {avg_val:.4f}")

    if avg_val < best_val:
        best_val = avg_val
        torch.save({'model_state_dict': model.state_dict(), 'num_outputs': num_outputs}, best_path)
        print('  -> Saved new best checkpoint to:', best_path)


In [ ]:
# Optional: compute macro F1 and ROC AUC on validation set
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score

model.eval()
probs_list = []
y_list = []
with torch.no_grad():
    for xb, yb in valid_loader:
        xb = xb.to(device)
        logits = model(ecg=xb, tabular=None)
        probs = torch.sigmoid(logits).cpu().numpy()
        probs_list.append(probs)
        y_list.append(yb.numpy())

probs = np.concatenate(probs_list, axis=0)
y_true = np.concatenate(y_list, axis=0)

preds = (probs >= 0.5).astype(int)
f1_macro = f1_score(y_true, preds, average='macro', zero_division=0)
try:
    auc_macro = roc_auc_score(y_true, probs, average='macro')
except ValueError:
    auc_macro = float('nan')

print('Validation metrics:')
print('  F1 macro:', f1_macro)
print('  ROC AUC macro:', auc_macro)
